In [1]:
import ROOT
ROOT.EnableImplicitMT(16)
import pandas as pd
import libPy
weights = [
    "hw_nominal",     # nominal MC weight.                  scalar
    "hw_alphaS_up",   # up alpha_s variaton for PHD4LHC;    scalar
    "hw_alphaS_dn",   # down alpha_s variaton for PHD4LHC;  scalar
    "hw_pdf4lhc_unc", # 30 Eigen variation for PHD4LHC      vector
    "hw_qcd",         # muR/muF variation for the given MC; vector
]

In [2]:
dfs = {}
dfs['ALL']      = ROOT.RDataFrame("tree", ["ntuples/mc20_wph_hyy_stxs.root", "ntuples/mc20_wmh_hyy_stxs.root"])
dfs['UNKNOWN']  = dfs['ALL'].Filter("HTXS_Stage1_2_Fine_Category_pTjet30 == 0")

for val, category in libPy.stage_1_2_fine['vbf'].items():
    dfs[category] = dfs['ALL'].Filter(f"HTXS_Stage1_2_Fine_Category_pTjet30 == {val}")
for val, category in libPy.stage_1_2_fine['wh'].items():
    dfs[category] = dfs['ALL'].Filter(f"HTXS_Stage1_2_Fine_Category_pTjet30 == {val}")

# # determine the length of the vector weights, which should be the same for all events
# len_hw_pdf4lhc_unc = set(dfs['ALL'].Range(10).Define('len_hw_pdf4lhc_unc', 'hw_pdf4lhc_unc.size()').AsNumpy(['len_hw_pdf4lhc_unc'])['len_hw_pdf4lhc_unc'])
# len_hw_qcd         = set(dfs['ALL'].Range(10).Define('len_hw_qcd', 'hw_qcd.size()').AsNumpy(['len_hw_qcd'])['len_hw_qcd'])
# assert (len(len_hw_pdf4lhc_unc) == 1 and len(len_hw_qcd) == 1)
# len_hw_pdf4lhc_unc = list(len_hw_pdf4lhc_unc)[0]
# len_hw_qcd         = list(len_hw_qcd)[0]
len_hw_pdf4lhc_unc, len_hw_qcd = 30, 8

In [3]:
weight_dict = {}
futures = []
for slice, df in dfs.items():
    weight_dict[slice] = {}
    for weight in weights:
        if weight == "hw_pdf4lhc_unc":
            for i in range(len_hw_pdf4lhc_unc):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        elif weight == "hw_qcd":
            for i in range(len_hw_qcd):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        else:
            weight_dict[slice][weight] = df.Filter(f"{weight} == {weight}").Sum(weight)
            futures.append(weight_dict[slice][weight])
ROOT.RDF.RunGraphs(futures)

1

In [4]:
for slice, weight_sum_dict in weight_dict.items():
    for weight_name, weight_sum in weight_sum_dict.items():
        weight_dict[slice][weight_name] = weight_sum.GetValue()

In [5]:
pdf = pd.DataFrame(weight_dict)
pdf = pdf.apply(lambda row : (row / row['ALL']), axis=1)
ratio_pdf = pdf.apply(lambda row : row / pdf.iloc[0], axis=1)
# pdf.columns = [col + '_acc' for col in pdf.columns]
# pdf[[col.split('_')[0] + '_xs' for col in pdf.columns]] = pdf.apply(lambda row : row * xs, axis=1)
# pdf = pd.concat([pdf, ratio_pdf.add_suffix('_ratio')], axis=1)
pdf.to_csv("res_stxs/run2_wh_stxs.csv", index=True)

In [6]:
import pandas as pd
import libPy
pdf = pd.read_csv("res_stxs/run3_wh_stxs.csv", index_col=0)

official = {key.upper() : val for key, val in libPy.official_1_2_fine_run2['wh'].items()}
official['UNKNOWN'] = 100 - sum(official.values())
mine = pdf.iloc[0] * 100
mine.index = [index.upper() for index in mine.index]
comp_df = pd.DataFrame.from_dict(official, orient='index')
comp_df.columns = ['official_run2']
comp_df['mine'] = mine
comp_df['diff'] = comp_df['mine'] - comp_df['official_run2']
comp_df['diff_pct'] = (comp_df['diff'] / comp_df['official_run2'] * 100)
comp_df

,official_run2,mine,diff,diff_pct
QQ2HLNU_FWDH,3.944460e+00,3.932330,-0.012130,-3.075220e-01
QQ2HLNU_PTV_0_75_0J,1.046010e+01,10.416710,-0.043390,-4.148150e-01
QQ2HLNU_PTV_75_150_0J,5.981320e+00,6.010167,0.028847,4.822794e-01
QQ2HLNU_PTV_150_250_0J,1.657520e+00,1.669066,0.011546,6.966125e-01
QQ2HLNU_PTV_250_400_0J,3.933200e-01,0.389954,-0.003366,-8.557997e-01
QQ2HLNU_PTV_GT400_0J,8.091950e-02,0.080489,-0.000430,-5.316972e-01
QQ2HLNU_PTV_0_75_1J,3.187600e+00,3.145827,-0.041773,-1.310469e+00
QQ2HLNU_PTV_75_150_1J,2.377430e+00,2.376168,-0.001262,-5.306268e-02
QQ2HLNU_PTV_150_250_1J,8.463260e-01,0.833214,-0.013112,-1.549241e+00
QQ2HLNU_PTV_250_400_1J,2.229820e-01,0.230371,0.007389,3.313927e+00
